In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

# Reload and rerun pipeline
data = sio.loadmat('../data/S17_E1_A1.mat')
emg = data['emg']
labels = data['restimulus']
fs = 2000

def bandpass_filter(signal, lowcut=20, highcut=500, fs=2000, order=4):
    nyq = fs / 2
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal)

def mav(window): return np.mean(np.abs(window))
def rms(window): return np.sqrt(np.mean(window**2))
def zero_crossing_rate(window, threshold=1e-6):
    zc = 0
    for i in range(1, len(window)):
        if ((window[i] >= threshold and window[i-1] < threshold) or
            (window[i] < -threshold and window[i-1] >= -threshold)):
            zc += 1
    return zc
def waveform_length(window): return np.sum(np.abs(np.diff(window)))

def extract_features(window):
    features = []
    for ch in range(window.shape[1]):
        ch_signal = window[:, ch]
        features.extend([mav(ch_signal), rms(ch_signal),
                         zero_crossing_rate(ch_signal), waveform_length(ch_signal)])
    return np.array(features)

# Filter all channels
emg_filtered = np.zeros_like(emg)
for ch in range(12):
    emg_filtered[:, ch] = bandpass_filter(emg[:, ch])

# Extract features
window_samples = int(0.2 * fs)
step_samples = int(0.1 * fs)
X, y = [], []

for start in range(0, len(emg_filtered) - window_samples, step_samples):
    end = start + window_samples
    window = emg_filtered[start:end, :]
    label = labels[start + window_samples//2, 0]
    X.append(extract_features(window))
    y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Dataset: {X.shape[0]} windows, {X.shape[1]} features, {len(np.unique(y))} classes")

In [ ]:
# Train and evaluate LDA with cross-validation
lda = LinearDiscriminantAnalysis()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(lda, X, y, cv=cv, scoring='accuracy')

print(f"Cross-validation accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"Per fold: {[f'{s:.3f}' for s in scores]}")

In [ ]:
# Fit on full data for confusion matrix visualization
lda.fit(X, y)
y_pred = lda.predict(X)

cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                               display_labels=[f'G{i}' for i in range(18)])

fig, ax = plt.subplots(figsize=(12, 10))
disp.plot(ax=ax, colorbar=True, cmap='Blues')
plt.title('LDA Confusion Matrix - EMG Gesture Classification')
plt.tight_layout()
plt.show()